# Case Study 6: Waste and Trash Detection

**Goal:** detect waste items in photos, draw a bounding box around each one, name its category (Bottle, Bottle Cap, Can, Cardboard, Cigarette Box, Cigarette Butt, Cup, Plastic), and **compare three object-detection models** on test images the models have never seen.

| # | Model | Approach | Why we chose it |
|---|-------|----------|-----------------|
| 1 | **YOLOv8m** | One-stage CNN, anchor-free, needs NMS | Baseline model given by the instructor |
| 2 | **YOLO11m** | Newer one-stage CNN in the same family, adds an attention block | Checks whether a newer architecture helps on a small dataset |
| 3 | **RT-DETR-L** | Transformer-based detector, end-to-end, **no NMS** | A different way of doing detection altogether |

**Pipeline**

```
Kaggle zip ─► pick the 163 ORIGINAL images + Pascal VOC boxes ─► leak-free 70/15/15 split
          ─► YOLO-format dataset ─► train 3 models (transfer learning from COCO)
          ─► test-set metrics (Precision, Recall, mAP) ─► error analysis (correct / missed / false)
          ─► comparison table + figures + summary for the report
```

**How to run:** select the `.venv` kernel (top-right in VS Code), then run the cells from top to bottom with **Shift + Enter**. Each code cell has a note above it explaining what it does. Everything is saved to disk:

| Folder | Contents |
|--------|----------|
| `data/trash_yolo/` | the clean dataset we build (images + labels + `data.yaml`) |
| `runs/train/<model>/` | trained weights (`weights/best.pt`), training curves, `results.csv` |
| `runs/test/<model>/` | test-set confusion matrix and PR curves |
| `results/` | CSV tables and `summary.md` (numbers for the report) |
| `report_figures/` | PNG figures to paste into the report |

> If the kernel crashes or you close VS Code, just run everything again. Models that are already trained are **not** trained a second time.

---
## 1. Setup

### 1.1 Imports
| Library | What we use it for |
|---------|--------------------|
| `torch` | Deep-learning engine that runs the models on the GPU or CPU |
| `ultralytics` | Training, validation and prediction for YOLOv8, YOLO11 and RT-DETR through one API |
| `xml.etree` | Reading the Pascal VOC `.xml` annotation files |
| `PIL` | Opening, rotating and resizing images |
| `numpy`, `pandas` | Arrays and tables |
| `matplotlib`, `seaborn` | Plots for the report |

In [ ]:
import os, sys, json, time, random, shutil, zipfile, urllib.request
import xml.etree.ElementTree as ET
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import yaml
from PIL import Image, ImageOps

import torch
import ultralytics
from ultralytics import YOLO, RTDETR
from IPython.display import display, Markdown, Image as IPyImage

sns.set_theme(style="whitegrid")
print("Python     :", sys.version.split()[0])
print("PyTorch    :", torch.__version__)
print("Ultralytics:", ultralytics.__version__)
print("GPU        :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none -> training will run on the CPU")

### 1.2 Configuration
Every setting is defined in this one cell, so it's easy to explain and to change.

- **Paths:** everything lives inside the project folder (the folder this notebook is in).
- **Seed:** fixes the random number generators so the split and training are repeatable.
- **Hardware settings:** chosen automatically.
  - **GPU:** up to 80 epochs at 640×640 px. Early stopping usually ends training sooner.
  - **CPU:** much slower, so each model gets a **time cap of about 24 minutes** (`TIME_HOURS`). Ultralytics then works out how many epochs fit into that time. Images are 416 px to save time.
- **Epoch:** one full pass over the training images. **Batch:** how many images the model sees before it updates its weights. **imgsz:** the size images are resized to before going into the network.
- **Patience:** early stopping. If the validation mAP hasn't improved for this many epochs, training stops and the best epoch is kept.

In [ ]:
# ---------- paths ----------
PROJECT_DIR = Path.cwd()                              # folder that contains this notebook
RAW_DIR     = PROJECT_DIR / "data" / "raw"            # the unzipped Kaggle dataset
YOLO_DIR    = PROJECT_DIR / "data" / "trash_yolo"     # the clean YOLO-format dataset we build
RUNS_DIR    = PROJECT_DIR / "runs"                    # training / testing outputs
RESULTS_DIR = PROJECT_DIR / "results"                 # CSV tables + summary.md
FIG_DIR     = PROJECT_DIR / "report_figures"          # PNG figures for the report
for d in (RAW_DIR, RESULTS_DIR, FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)

# ---------- reproducibility ----------
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# ---------- hardware-dependent training settings ----------
GPU    = torch.cuda.is_available()
DEVICE = 0 if GPU else "cpu"
if GPU:
    EPOCHS, IMGSZ, BATCH = 80, 640, 8
    TIME_HOURS = None        # no time cap -> train for EPOCHS (early stopping may finish sooner)
else:
    EPOCHS, IMGSZ, BATCH = 40, 416, 4
    TIME_HOURS = 0.4         # ~24 min per model on CPU (overrides EPOCHS)
PATIENCE = 20                # early stopping patience (epochs)
WORKERS  = 2                 # data-loading processes; set to 0 if you get a "DataLoader worker" error
# GPU "CUDA out of memory" error?  -> set BATCH = 4 (or 2) and re-run this cell + the training cell

# ---------- data ----------
SPLIT    = {"train": 0.70, "val": 0.15, "test": 0.15}
MAX_SIDE = 1280              # large phone photos are shrunk to this longest side (labels are relative, so they stay correct)
IMG_EXT  = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

# ---------- error analysis ----------
CONF_THR = 0.25              # a prediction must have at least this confidence to count as a detection
IOU_THR  = 0.50              # a detection is correct if IoU with a true box >= 0.5 and the class matches

print(f"device={DEVICE} | epochs={EPOCHS} | imgsz={IMGSZ} | batch={BATCH} | time cap per model (h)={TIME_HOURS}")

---
## 2. Get the dataset

If you've already unzipped the Kaggle zip into `data/raw/`, this cell does nothing. If not, it downloads the public zip without a Kaggle login and extracts it.
If the download fails (for example on a blocked network), download the zip from the Kaggle page in your browser, put it at `data/trash.zip` and run the cell again.

In [ ]:
KAGGLE_URL = "https://www.kaggle.com/api/v1/datasets/download/pandapowr/labeled-trash-dataset-pascal-voc-coco-and-yolo"
zip_path = PROJECT_DIR / "data" / "trash.zip"

if any(RAW_DIR.rglob("*.xml")):
    print("Dataset already present in", RAW_DIR)
else:
    if not zip_path.exists():
        print("Downloading dataset (may take a few minutes)...")
        req = urllib.request.Request(KAGGLE_URL, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req) as r, open(zip_path, "wb") as f:
            shutil.copyfileobj(r, f)
    print("Extracting...")
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(RAW_DIR)
    print("Done ->", RAW_DIR)

---
## 3. Understand the raw data and pick what to train on

### 3.1 What's inside the zip
The zip holds several versions of the same photos (original, resized, augmented, pre-split), each with annotations in three formats. The table below lists every folder and what kind of files it contains.

In [ ]:
rows = []
for d in sorted({p.parent for p in RAW_DIR.rglob("*") if p.is_file()}):
    if "__MACOSX" in d.parts:
        continue
    files = [f for f in d.iterdir() if f.is_file()]
    rows.append({"folder": str(d.relative_to(RAW_DIR)),
                 "images":   sum(f.suffix.lower() in IMG_EXT for f in files),
                 "voc_xml":  sum(f.suffix.lower() == ".xml" for f in files),
                 "yolo_txt": sum(f.suffix.lower() == ".txt" for f in files),
                 "json":     sum(f.suffix.lower() == ".json" for f in files)})
pd.set_option("display.max_colwidth", 150)
display(pd.DataFrame(rows))

### 3.2 Which data we use, and why (be ready to explain this)

**We train on the 163 ORIGINAL images with their Pascal VOC annotations.**

1. **We avoid data leakage.** The 4,923 "augmented" images are edited copies of the 163 originals (flipped, rotated, recoloured and so on). If one copy of a photo ends up in the training set and another copy of the same photo ends up in the test set, the model has in effect already seen the test photo, and the scores come out too high. The brief says to test on **unseen** images, so we split the *originals* first. Ultralytics then augments only the training images, on the fly, in every epoch (mosaic, flips, colour changes, scaling).
2. **Pascal VOC XML stores class *names* as text** (for example `<name>Bottle</name>`). With YOLO `.txt` files you only get numbers and have to trust that the class-ID order is right. Using VOC removes that risk.

The next cell finds the folder with the original annotations automatically, whatever the folders are called. If it picks the wrong one, set `SOURCE_DIR` to the correct folder and run it again.

In [ ]:
SOURCE_DIR = None        # None = auto-detect. Or force a folder, e.g. RAW_DIR / "Original Data"

ORIGINAL_KW = ("original", "orijinal", "orjinal")
RESIZED_KW  = ("resiz",)
EXCLUDE_KW  = ("augment", "split", "artir", "artır", "bolun", "bölün")

all_xml = sorted(p for p in RAW_DIR.rglob("*.xml")
                 if "__MACOSX" not in p.parts and not p.name.startswith("._"))
if not all_xml:
    raise FileNotFoundError(f"No Pascal VOC .xml files under {RAW_DIR}. Did you extract the zip there?")

# folder names that ALL xml files share (e.g. the zip's top folder) are ignored when matching keywords
common = os.path.commonpath([str(p.parent) for p in all_xml])
def dir_names(p):
    return [s.lower() for s in p.parent.relative_to(common).parts] if str(p.parent) != common else []
def has_kw(p, kws):
    return any(k in d for d in dir_names(p) for k in kws)

def choose_annotation_set():
    if SOURCE_DIR is not None:
        root = Path(SOURCE_DIR)
        return root, sorted(root.rglob("*.xml")), "manual"
    usable = [x for x in all_xml if not has_kw(x, EXCLUDE_KW)] or all_xml
    for label, kws in (("original", ORIGINAL_KW), ("resized", RESIZED_KW)):
        hits = [x for x in usable if has_kw(x, kws)]
        if hits:
            parts = hits[0].relative_to(RAW_DIR).parts
            i = next(i for i, s in enumerate(parts) if any(k in s.lower() for k in kws))
            root = RAW_DIR.joinpath(*parts[: i + 1])
            return root, [x for x in hits if root in x.parents], label
    by_dir = Counter(x.parent for x in usable)                    # last resort: folder with ~163 xml files
    best = min(by_dir, key=lambda d: abs(by_dir[d] - 163))
    print("WARNING: no 'original' folder found. Check the folder below really holds the original images, otherwise set SOURCE_DIR.")
    return best.parent, sorted(best.glob("*.xml")), "fallback (folder with ~163 xml files)"

SRC_ROOT, src_xmls, how = choose_annotation_set()
print(f"Using {how} annotations from: {SRC_ROOT}")
print(f"XML files found: {len(src_xmls)}")

### 3.3 Read the XML annotations and match each one to its image
A Pascal VOC file looks like this (one `<object>` per box, in **pixel** coordinates):
```xml
<annotation>
  <filename>IMG_0012.jpg</filename>
  <size><width>3024</width><height>4032</height></size>
  <object><name>Bottle</name>
    <bndbox><xmin>812</xmin><ymin>1330</ymin><xmax>1504</xmax><ymax>2811</ymax></bndbox>
  </object>
</annotation>
```
The cell below reads every file and finds the matching image. It also tidies class names, so that `"bottle "` and `"Bottle"` count as one class.

In [ ]:
def parse_voc(xml_path):
    root = ET.parse(xml_path).getroot()
    size = root.find("size")
    w = int(float(size.findtext("width", "0") or 0)) if size is not None else 0
    h = int(float(size.findtext("height", "0") or 0)) if size is not None else 0
    objs = []
    for o in root.iter("object"):
        bb = o.find("bndbox")
        if bb is None:
            continue
        x1, y1, x2, y2 = (float(bb.findtext(k)) for k in ("xmin", "ymin", "xmax", "ymax"))
        objs.append([o.findtext("name", "").strip(), x1, y1, x2, y2])
    return (root.findtext("filename") or "").strip(), w, h, objs

# index every image under the chosen folder by lower-case file stem
img_index = defaultdict(list)
for p in SRC_ROOT.rglob("*"):
    if p.suffix.lower() in IMG_EXT and "__MACOSX" not in p.parts:
        img_index[p.stem.lower()].append(p)

def find_image(xml_path, filename):
    for stem in (xml_path.stem, Path(filename).stem):
        same_dir = [p for p in xml_path.parent.iterdir() if p.stem.lower() == stem.lower() and p.suffix.lower() in IMG_EXT]
        cands = same_dir or img_index.get(stem.lower(), [])
        if cands:
            return sorted(cands)[0]
    return None

norm = lambda s: " ".join(s.replace("_", " ").replace("-", " ").lower().split())
spellings = defaultdict(Counter)
image_info, missing, broken = {}, [], []
for x in src_xmls:
    if x.stem in image_info:                       # same annotation stored twice -> keep one
        continue
    try:
        fname, w, h, objs = parse_voc(x)
    except Exception:
        broken.append(x); continue
    img_path = find_image(x, fname)
    if img_path is None:
        missing.append(x); continue
    for o in objs:
        spellings[norm(o[0])][o[0]] += 1
    image_info[x.stem] = {"path": img_path, "xml_w": w, "xml_h": h, "objects": objs}

# one clean display name per class (the most common spelling), ids in alphabetical order
CANON = {k: c.most_common(1)[0][0] for k, c in spellings.items()}
for info in image_info.values():
    for o in info["objects"]:
        o[0] = CANON[norm(o[0])]
CLASS_NAMES = sorted(set(CANON.values()))
CLASS_TO_ID = {c: i for i, c in enumerate(CLASS_NAMES)}

print(f"Images with annotations: {len(image_info)}  | XML without image: {len(missing)} | unreadable XML: {len(broken)}")
print(f"{len(CLASS_NAMES)} classes: {CLASS_NAMES}")
if len(image_info) < 0.9 * len(src_xmls):
    print("WARNING: many XML files had no matching image. Check the folder table above and set SOURCE_DIR.")
if len(CLASS_NAMES) != 8:
    print("NOTE: the dataset page says 8 classes. Check the list above for typos or duplicate spellings.")

---
## 4. Exploratory data analysis (EDA)

Before training, we look at the data. Each check below relates to one of the challenges listed in the brief:
- **Class balance:** rare classes usually get a lower AP, so we report AP for each class separately.
- **Object sizes** ("objects can have different sizes"): small items such as cigarette butts and bottle caps are the hardest to detect.
- **Overlap** ("waste objects may overlap"): the share of boxes that overlap another box.

In [ ]:
def box_iou(a, b):
    # IoU between every box in a (N,4) and every box in b (M,4); boxes are [x1, y1, x2, y2]
    a, b = np.asarray(a, float).reshape(-1, 4), np.asarray(b, float).reshape(-1, 4)
    ix1 = np.maximum(a[:, None, 0], b[None, :, 0]); iy1 = np.maximum(a[:, None, 1], b[None, :, 1])
    ix2 = np.minimum(a[:, None, 2], b[None, :, 2]); iy2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(ix2 - ix1, 0, None) * np.clip(iy2 - iy1, 0, None)
    area_a = (a[:, 2] - a[:, 0]) * (a[:, 3] - a[:, 1]); area_b = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    return inter / (area_a[:, None] + area_b[None, :] - inter + 1e-9)

rows = []
for stem, info in image_info.items():
    W, H = (info["xml_w"], info["xml_h"]) if min(info["xml_w"], info["xml_h"]) > 0 else Image.open(info["path"]).size
    boxes = np.array([o[1:] for o in info["objects"]]).reshape(-1, 4)
    ious = box_iou(boxes, boxes); np.fill_diagonal(ious, 0)
    for k, (c, x1, y1, x2, y2) in enumerate(info["objects"]):
        rows.append({"image": stem, "class": c,
                     "rel_size": np.sqrt(max(x2 - x1, 0) * max(y2 - y1, 0) / (W * H)),   # sqrt of area fraction
                     "aspect": (x2 - x1) / max(y2 - y1, 1e-6),
                     "overlaps": bool((ious[k] > 0).any())})
obj_df = pd.DataFrame(rows)
per_img = obj_df.groupby("image").size()

fig, ax = plt.subplots(1, 3, figsize=(17, 4.5))
obj_df["class"].value_counts().plot.barh(ax=ax[0], color="#4C72B0"); ax[0].invert_yaxis()
ax[0].set_title("Objects per class"); ax[0].set_xlabel("count")
ax[1].hist(per_img, bins=range(1, per_img.max() + 2), color="#55A868", align="left")
ax[1].set_title("Objects per image"); ax[1].set_xlabel("objects"); ax[1].set_ylabel("images")
sns.boxplot(data=obj_df, y="class", x="rel_size", ax=ax[2], color="#C44E52", order=CLASS_NAMES)
ax[2].set_title("Box size by class (sqrt of area fraction)"); ax[2].set_xlabel("0.1 = box covers 1% of the image")
plt.tight_layout(); plt.savefig(FIG_DIR / "01_dataset_overview.png", dpi=150); plt.show()

print(f"Images: {len(image_info)} | objects: {len(obj_df)} | average objects/image: {per_img.mean():.2f}")
print(f"Objects overlapping at least one other box: {obj_df.overlaps.mean():.1%}")
print(f"Images containing overlapping objects: {obj_df.groupby('image').overlaps.any().mean():.1%}")
print(f"Small objects (<1% of image area): {(obj_df.rel_size < 0.1).mean():.1%}")

---
## 5. Train / validation / test split

- **Train (70%):** the model learns from these images.
- **Validation (15%):** checked after every epoch to pick the best epoch and to trigger early stopping. Because we make decisions based on it, it isn't truly "unseen".
- **Test (15%):** used only once, at the very end. **These are the unseen images the brief asks for.**

We split **by image**, so every box from one photo lands in the same split. With only 163 images, a purely random split can leave a rare class out of the test set. The cell therefore tries 500 random splits and keeps the one whose class mix in each split is closest to the full dataset, and where every class appears in validation and test.

In [ ]:
def make_split(stems, ratios, tries=500, seed=SEED):
    rng = np.random.default_rng(seed)
    cls_of = {s: [o[0] for o in image_info[s]["objects"]] for s in stems}
    total = Counter(c for s in stems for c in cls_of[s]); n_total = sum(total.values())
    n = len(stems); n_tr = round(ratios["train"] * n); n_va = round(ratios["val"] * n)
    best, best_score = None, float("inf")
    for _ in range(tries):
        perm = [str(s) for s in rng.permutation(stems)]
        sp = {"train": perm[:n_tr], "val": perm[n_tr:n_tr + n_va], "test": perm[n_tr + n_va:]}
        score = 0.0
        for name, ss in sp.items():
            cnt = Counter(c for s in ss for c in cls_of[s]); n_cnt = max(sum(cnt.values()), 1)
            for c in total:
                score += abs(cnt[c] / n_cnt - total[c] / n_total)
                if name != "train" and cnt[c] == 0:
                    score += 1.0                      # heavy penalty: class missing from val/test
        if score < best_score:
            best, best_score = sp, score
    return best

splits = make_split(sorted(image_info), SPLIT)
split_table = pd.DataFrame({k: Counter(o[0] for s in v for o in image_info[s]["objects"]) for k, v in splits.items()}).reindex(CLASS_NAMES).fillna(0).astype(int)
split_table.loc["IMAGES"] = [len(v) for v in splits.values()]
split_table.to_csv(RESULTS_DIR / "split_table.csv")
display(split_table)

---
## 6. Build the YOLO-format dataset

Ultralytics expects this layout:
```
data/trash_yolo/
├── images/train|val|test/IMG_0012.jpg
├── labels/train|val|test/IMG_0012.txt     <- one line per box
└── data.yaml                                <- paths + class names
```
Each label line is `class_id x_center y_center width height`, **normalised to 0–1** by the image width and height. For example, `2 0.51 0.43 0.22 0.35` means class 2, centred at 51% across and 43% down the image.

Two details the code takes care of:
- **Phone photo rotation (EXIF).** Phones often store a photo sideways and add a "rotate me" tag. If the XML's width and height are swapped compared with the stored pixels, the annotator saw the rotated view, so we apply that rotation. We then save the image **without** the tag, so every library sees the same orientation as the boxes.
- **Resizing.** Large photos are shrunk to a longest side of `MAX_SIDE`. The labels are relative (0–1), so they don't change.

In [ ]:
def build_yolo_dataset():
    if YOLO_DIR.exists():
        shutil.rmtree(YOLO_DIR)
    n_rot = n_dropped = 0
    for split, stems in splits.items():
        (YOLO_DIR / "images" / split).mkdir(parents=True)
        (YOLO_DIR / "labels" / split).mkdir(parents=True)
        for s in stems:
            info = image_info[s]
            img = Image.open(info["path"])
            raw_w, raw_h = img.size
            xw, xh = info["xml_w"], info["xml_h"]
            if xw > 0 and xh > 0 and (xw, xh) == (raw_h, raw_w) and raw_w != raw_h:
                img = ImageOps.exif_transpose(img); n_rot += 1       # annotator saw the rotated view
            W, H = (xw, xh) if (xw > 0 and xh > 0) else img.size   # coordinate system of the boxes
            img = img.convert("RGB")
            scale = min(1.0, MAX_SIDE / max(img.size))
            if scale < 1:
                img = img.resize((round(img.width * scale), round(img.height * scale)), Image.LANCZOS)
            img.save(YOLO_DIR / "images" / split / f"{s}.jpg", quality=95)   # saved without EXIF

            lines = []
            for c, x1, y1, x2, y2 in info["objects"]:
                x1, x2 = sorted((min(max(x1, 0), W), min(max(x2, 0), W)))
                y1, y2 = sorted((min(max(y1, 0), H), min(max(y2, 0), H)))
                if x2 - x1 < 1 or y2 - y1 < 1:
                    n_dropped += 1; continue                          # degenerate box
                lines.append(f"{CLASS_TO_ID[c]} {(x1 + x2) / 2 / W:.6f} {(y1 + y2) / 2 / H:.6f} "
                             f"{(x2 - x1) / W:.6f} {(y2 - y1) / H:.6f}")
            (YOLO_DIR / "labels" / split / f"{s}.txt").write_text("\n".join(lines), encoding="utf-8")
    print(f"Rotated by EXIF: {n_rot} | degenerate boxes dropped: {n_dropped}")

build_yolo_dataset()

DATA_YAML = YOLO_DIR / "data.yaml"
data_cfg = {"path": YOLO_DIR.as_posix(), "train": "images/train", "val": "images/val", "test": "images/test",
            "names": {i: n for i, n in enumerate(CLASS_NAMES)}}
DATA_YAML.write_text(yaml.safe_dump(data_cfg, sort_keys=False), encoding="utf-8")
print(DATA_YAML.read_text())

### 6.1 Sanity check: draw the converted labels
This cell reads the labels **back from the YOLO files** and draws them on the images. If the boxes sit neatly on the objects, the whole conversion worked. Always look at your labels before training.

In [ ]:
PALETTE = plt.cm.tab10(np.linspace(0, 1, 10))

def read_yolo_labels(img_path):
    lbl = YOLO_DIR / "labels" / img_path.parent.name / f"{img_path.stem}.txt"
    W, H = Image.open(img_path).size
    txt = lbl.read_text().strip() if lbl.exists() else ""
    arr = np.array([l.split() for l in txt.splitlines()], float).reshape(-1, 5)
    cls = arr[:, 0].astype(int)
    cx, cy, bw, bh = arr[:, 1] * W, arr[:, 2] * H, arr[:, 3] * W, arr[:, 4] * H
    return np.stack([cx - bw / 2, cy - bh / 2, cx + bw / 2, cy + bh / 2], 1), cls, (W, H)

def draw_boxes(ax, boxes, labels, color, ls="-", lw=2):
    for (x1, y1, x2, y2), lab in zip(boxes, labels):
        ax.add_patch(mpatches.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, ec=color, lw=lw, ls=ls))
        ax.text(x1, y1, lab, color="white", fontsize=7, va="bottom", bbox=dict(fc=color, ec="none", pad=1, alpha=0.85))

sample = random.Random(SEED).sample(sorted((YOLO_DIR / "images" / "train").glob("*.jpg")), 6)
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
for ax, p in zip(axes.flat, sample):
    boxes, cls, _ = read_yolo_labels(p)
    ax.imshow(Image.open(p)); ax.axis("off"); ax.set_title(p.name, fontsize=9)
    for b, c in zip(boxes, cls):
        draw_boxes(ax, [b], [CLASS_NAMES[c]], PALETTE[c % 10])
plt.tight_layout(); plt.savefig(FIG_DIR / "02_label_check.png", dpi=120); plt.show()

---
## 7. Train the three models

### How training works (applies to all three models)
- **Transfer learning:** each model starts from weights **pre-trained on COCO**, a dataset of 118k images covering 80 everyday objects, *bottle* and *cup* among them. The network already knows edges, textures and object shapes, so 115 training images are enough to adapt it to our 8 classes. Training from scratch on 115 images would fail.
- **On-the-fly augmentation (training images only):** *mosaic* stitches 4 images into one, which gives new backgrounds, scales and partly hidden objects. It is switched off in the last 10 epochs so the model finishes on realistic images. Random flips, HSV colour shifts, scaling and translation are applied too. This helps with the "different shapes, sizes and backgrounds" challenge.
- **Each epoch:** the model predicts boxes for each batch, the **loss** measures how wrong those predictions are, and back-propagation adjusts the weights to reduce the loss. The model is then scored on the validation set. The best epoch (by validation mAP) is saved as `best.pt`.
- **Losses:** YOLO uses a box loss (CIoU, which measures how well the boxes overlap), a classification loss (BCE) and DFL (which sharpens box edges). RT-DETR uses GIoU + L1 box losses and a classification loss, after pairing predictions with true boxes one-to-one (Hungarian matching).
- **Same settings for all three** (image size, epochs or time cap, batch, seed, data split) make it a **fair comparison**.

The helper below trains one model and records how long it took. If `best.pt` already exists, training is **skipped**, so re-running the notebook never wastes time. Set `RETRAIN = True` to force it.

In [ ]:
MODELS = {
    "YOLOv8m":   {"weights": "yolov8m.pt",  "cls": YOLO},     # given by the instructor
    "YOLO11m":   {"weights": "yolo11m.pt",  "cls": YOLO},
    "RT-DETR-L": {"weights": "rtdetr-l.pt", "cls": RTDETR},   # RT-DETR needs its own class (different predictor, no NMS)
}
RETRAIN    = False
TIMES_FILE = RESULTS_DIR / "train_times.json"

def best_weights(name):
    return RUNS_DIR / "train" / name / "weights" / "best.pt"

def train_model(name):
    if best_weights(name).exists() and not RETRAIN:
        print(f"{name}: already trained ({best_weights(name)}) -> skipping. Set RETRAIN = True to train again.")
        return
    cfg = MODELS[name]
    model = cfg["cls"](cfg["weights"])            # loads COCO-pretrained weights (downloads if missing)
    t0 = time.time()
    model.train(data=str(DATA_YAML), epochs=EPOCHS, time=TIME_HOURS, imgsz=IMGSZ, batch=BATCH,
                device=DEVICE, workers=WORKERS, patience=PATIENCE, seed=SEED, deterministic=True,
                cache="ram", project=str(RUNS_DIR / "train"), name=name, exist_ok=True, plots=True)
    minutes = (time.time() - t0) / 60
    times = json.loads(TIMES_FILE.read_text()) if TIMES_FILE.exists() else {}
    times[name] = round(minutes, 1)
    TIMES_FILE.write_text(json.dumps(times, indent=2))
    print(f"\n{name}: finished in {minutes:.1f} min -> {best_weights(name)}")

### 7.1 Model 1: YOLOv8m (baseline, given by the instructor)
**YOLO = "You Only Look Once":** a *one-stage* detector. A single pass through the network predicts every box and class at once, which makes it fast.
- **Backbone** (CSPDarknet with **C2f** blocks) extracts features. **Neck** (PAN-FPN) merges features at 3 scales (strides 8/16/32), so it can find small, medium and large objects.
- **Head:** *decoupled* (separate branches for "what is it" and "where is it") and *anchor-free* (it predicts box edges directly from each grid point, with no predefined anchor-box shapes).
- **NMS (Non-Maximum Suppression):** the network produces many overlapping boxes for the same object. NMS keeps the highest-confidence box and removes the others that overlap it heavily.
- Size: **25.9 M parameters, 78.9 GFLOPs**, COCO mAP50-95 = 50.2. "m" = medium (the family goes n, s, m, l, x).

In [ ]:
train_model("YOLOv8m")

### 7.2 Model 2: YOLO11m (newer generation, same one-stage approach)
YOLO11 (Ultralytics, 2024) is the successor to YOLOv8. It keeps the same anchor-free head and NMS, and changes the building blocks:
- **C3k2** blocks replace C2f and are a more efficient CSP bottleneck.
- **C2PSA** is a small *attention* block after SPPF. It helps the network focus on the important regions of the image, which can help with cluttered and overlapping objects.
- Size: **20.1 M parameters, 68.0 GFLOPs**, COCO mAP50-95 = 51.5. It has *fewer* parameters than YOLOv8m and scores *higher* on COCO.
- **What this comparison tests:** do architecture improvements measured on a large dataset still pay off on our small one?

In [ ]:
train_model("YOLO11m")

### 7.3 Model 3: RT-DETR-L (transformer-based, a different approach)
**RT-DETR** (Real-Time DEtection TRansformer, Baidu 2023) works differently from YOLO:
- A CNN backbone (**HGNetv2**) extracts features. A **hybrid encoder** then combines *self-attention* on the deepest feature map (AIFI) with CNN-based fusion across scales (CCFM).
- A **transformer decoder** uses **300 "object queries"**. Each query is a learned slot that outputs at most one object.
- During training, predictions are paired with true boxes **one-to-one** (Hungarian matching). This teaches the model not to produce duplicates, so **no NMS is needed**, which makes it an *end-to-end* detector.
- Size: **32 M parameters, 110 GFLOPs**, COCO AP = 53.0.
- **Trade-off:** transformers usually need **more data and more epochs** to converge than CNNs. With 115 training images, that is exactly what this comparison tests.

In [ ]:
train_model("RT-DETR-L")

### 7.4 Training curves
Validation mAP for each epoch, all three models on one plot. Look at how fast each model learns (convergence speed) and whether it levels off (plateau) or gets worse (overfitting).

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(18, 4.5))
for name in MODELS:
    csv = RUNS_DIR / "train" / name / "results.csv"
    if not csv.exists():
        continue
    r = pd.read_csv(csv); r.columns = [c.strip() for c in r.columns]
    ax[0].plot(r["epoch"], r["metrics/mAP50(B)"], label=name)
    ax[1].plot(r["epoch"], r["metrics/mAP50-95(B)"], label=name)
    ax[2].plot(r["epoch"], r["metrics/recall(B)"], label=name)
for a, t in zip(ax, ["Validation mAP@0.5", "Validation mAP@0.5:0.95", "Validation recall"]):
    a.set_title(t); a.set_xlabel("epoch"); a.legend()
plt.tight_layout(); plt.savefig(FIG_DIR / "03_training_curves.png", dpi=150); plt.show()

---
## 8. Evaluate on the unseen TEST set

### The metrics, explained
- **IoU (Intersection over Union)** = overlap area ÷ combined area of a predicted box and a true box. 1 = perfect, 0 = no overlap.
- A prediction is a **True Positive (TP)** if it has the right class and IoU ≥ threshold with a true box that hasn't been matched yet. Otherwise it's a **False Positive (FP)**. A true box that no prediction matched is a **False Negative (FN)**, i.e. a missed detection.
- **Precision** = TP / (TP + FP): *of everything we detected, how much was right?* (few false alarms)
- **Recall** = TP / (TP + FN): *of all the waste in the images, how much did we find?* (few misses)
- **AP (Average Precision)** for one class = the area under its precision–recall curve as the confidence threshold moves from high to low. It summarises precision and recall at every threshold in one number.
- **mAP@0.5** = AP averaged over the 8 classes, with IoU ≥ 0.5 counting as correct. This is the "did it find the object" score.
- **mAP@0.5:0.95** = mAP averaged over IoU thresholds 0.50, 0.55, …, 0.95. This rewards **tight, accurate boxes**, so it is our measure of **bounding-box quality**.
- Ultralytics reports Precision and Recall at the confidence threshold that gives the best F1 score. mAP is computed over all thresholds.

The cell runs each trained model on the test split. It collects the overall metrics, the metrics for each class, the model size, and the confusion matrix and PR-curve plots.

In [ ]:
def load_best(name):
    return MODELS[name]["cls"](str(best_weights(name)))

test_rows, class_rows = [], []
for name in MODELS:
    if not best_weights(name).exists():
        print(f"{name}: not trained yet -> skipped"); continue
    model = load_best(name)
    _, n_params, _, gflops = model.info()          # prints a one-line summary; GFLOPs at 640 px
    m = model.val(data=str(DATA_YAML), split="test", imgsz=IMGSZ, batch=BATCH, device=DEVICE,
                  project=str(RUNS_DIR / "test"), name=name, exist_ok=True, plots=True, verbose=False)
    test_rows.append({"model": name, "params_M": n_params / 1e6, "GFLOPs": gflops,
                      "precision": m.box.mp, "recall": m.box.mr, "mAP50": m.box.map50, "mAP50-95": m.box.map})
    for i, c in enumerate(m.ap_class_index):
        class_rows.append({"model": name, "class": CLASS_NAMES[c], "P": m.box.p[i], "R": m.box.r[i],
                           "AP50": m.box.ap50[i], "AP50-95": m.box.ap[i]})

test_df = pd.DataFrame(test_rows).set_index("model")
class_df = pd.DataFrame(class_rows)
test_df.to_csv(RESULTS_DIR / "test_metrics.csv"); class_df.to_csv(RESULTS_DIR / "test_metrics_per_class.csv", index=False)
display(test_df.round(3))

### 8.1 AP for each class
Which categories are easy and which are hard? This usually lines up with the EDA: small or rare classes score lower.

In [ ]:
ap_table = class_df.pivot(index="class", columns="model", values="AP50").reindex(columns=list(test_df.index))
display(ap_table.round(3))
ax = ap_table.plot.barh(figsize=(10, 6), width=0.8)
ax.set_xlabel("AP@0.5 on test set"); ax.set_title("AP for each class on the test set"); ax.invert_yaxis(); ax.set_xlim(0, 1)
plt.tight_layout(); plt.savefig(FIG_DIR / "04_per_class_ap.png", dpi=150); plt.show()

### 8.2 Confusion matrices
Rows are the **predicted** class and columns the **true** class. The extra **"background"** row and column count false detections (the model predicted an object where there was none) and missed objects (a true object that was predicted as nothing). Large off-diagonal values show which classes the model mixes up, for example Plastic vs Bottle.

In [ ]:
for name in test_df.index:
    cm = sorted((RUNS_DIR / "test" / name).glob("confusion_matrix_normalized.png"))
    if cm:
        display(Markdown(f"**{name}**")); display(IPyImage(filename=str(cm[0]), width=650))

---
## 9. Error analysis: correct, missed and false detections

mAP condenses everything into one number. Here we look at the **actual detections** at a practical setting: confidence ≥ `CONF_THR` (0.25), counted as correct when IoU ≥ `IOU_THR` (0.5).

Each prediction is matched to the true boxes in order of confidence, highest first:
| Outcome | Meaning |
|---------|---------|
| **Correct (TP)** | right class, IoU ≥ 0.5 with a true box not already matched |
| **Missed (FN)** | a true object that no prediction matched |
| **False: wrong class** | the box is in the right place but the class is wrong (e.g. Plastic called Bottle) |
| **False: poor localisation** | right class, but the box overlaps the object too little (0.1 ≤ IoU < 0.5) |
| **False: duplicate** | a second box on an object that is already detected |
| **False: background** | the model "saw" waste where there is none |

This split shows *why* a model loses precision (the false types) or recall (the missed objects). The same loop also measures **inference time** for one image at a time, as a camera on a sorting line would process them.

In [ ]:
def load_gt(split="test"):
    gt = {}
    for p in sorted((YOLO_DIR / "images" / split).glob("*.jpg")):
        boxes, cls, (W, H) = read_yolo_labels(p)
        gt[p.name] = {"path": p, "boxes": boxes, "cls": cls, "W": W, "H": H}
    return gt

def analyse_image(g, pb, pc, pconf):
    # Match predictions to ground truth; returns one outcome per prediction + a matched flag per GT box.
    outcome = [""] * len(pb)
    matched = np.zeros(len(g["boxes"]), bool)
    ious = box_iou(pb, g["boxes"]) if len(pb) and len(g["boxes"]) else np.zeros((len(pb), len(g["boxes"])))
    for i in np.argsort(-pconf):
        same = g["cls"] == pc[i]
        cand = np.where(same & ~matched, ious[i], -1) if len(g["boxes"]) else np.array([-1])
        j = int(cand.argmax())
        if cand[j] >= IOU_THR:
            outcome[i] = "TP"; matched[j] = True
        elif len(g["boxes"]) and (np.where(same & matched, ious[i], 0) >= IOU_THR).any():
            outcome[i] = "FP: duplicate"
        elif len(g["boxes"]) and (np.where(~same, ious[i], 0) >= IOU_THR).any():
            outcome[i] = "FP: wrong class"
        elif len(g["boxes"]) and (np.where(same, ious[i], 0) >= 0.1).any():
            outcome[i] = "FP: poor localisation"
        else:
            outcome[i] = "FP: background"
    return outcome, matched

def size_bucket(box, W, H):
    s = np.sqrt((box[2] - box[0]) * (box[3] - box[1]) / (W * H))
    return "small" if s < 0.1 else ("medium" if s < 0.3 else "large")

GT_TEST = load_gt("test")
analysis, latency = {}, {}
for name in test_df.index:
    model = load_best(name)
    per_image, times = {}, []
    for fname, g in GT_TEST.items():
        r = model.predict(str(g["path"]), conf=CONF_THR, imgsz=IMGSZ, device=DEVICE, verbose=False)[0]
        times.append(r.speed["preprocess"] + r.speed["inference"] + r.speed["postprocess"])
        pb, pc, pconf = r.boxes.xyxy.cpu().numpy(), r.boxes.cls.cpu().numpy().astype(int), r.boxes.conf.cpu().numpy()
        out, matched = analyse_image(g, pb, pc, pconf)
        per_image[fname] = {"pb": pb, "pc": pc, "pconf": pconf, "outcome": out, "matched": matched}
    analysis[name], latency[name] = per_image, float(np.median(times))

# ---- summary counts per model ----
err_rows, cls_rows, size_rows = [], [], []
for name, per_image in analysis.items():
    cnt = Counter(o for d in per_image.values() for o in d["outcome"])
    fn = sum((~d["matched"]).sum() for d in per_image.values())
    tp, fp = cnt["TP"], sum(v for k, v in cnt.items() if k.startswith("FP"))
    err_rows.append({"model": name, "correct (TP)": tp, "missed (FN)": fn, "false (FP)": fp,
                     "FP: wrong class": cnt["FP: wrong class"], "FP: poor localisation": cnt["FP: poor localisation"],
                     "FP: duplicate": cnt["FP: duplicate"], "FP: background": cnt["FP: background"],
                     "precision@0.25": tp / max(tp + fp, 1), "recall@0.25": tp / max(tp + fn, 1),
                     "F1@0.25": 2 * tp / max(2 * tp + fp + fn, 1), "latency_ms": latency[name]})
    for fname, d in per_image.items():
        g = GT_TEST[fname]
        for k, c in enumerate(g["cls"]):
            cls_rows.append({"model": name, "class": CLASS_NAMES[c], "type": "TP" if d["matched"][k] else "FN"})
            size_rows.append({"model": name, "size": size_bucket(g["boxes"][k], g["W"], g["H"]), "found": d["matched"][k]})
        for c, o in zip(d["pc"], d["outcome"]):
            if o != "TP":
                cls_rows.append({"model": name, "class": CLASS_NAMES[c], "type": "FP"})

err_df = pd.DataFrame(err_rows).set_index("model")
err_df.to_csv(RESULTS_DIR / "error_analysis.csv")
display(err_df.round(3))

### 9.1 Errors by class and by object size
- **By class:** which categories are missed (FN) or cause false alarms (FP)?
- **By size:** recall for small, medium and large objects. This relates directly to the "different sizes" challenge.

In [ ]:
cls_err = pd.DataFrame(cls_rows).groupby(["class", "model", "type"]).size().unstack(fill_value=0)
cls_err = cls_err.reindex(columns=["TP", "FN", "FP"], fill_value=0)
display(cls_err.unstack("model").fillna(0).astype(int))
cls_err.to_csv(RESULTS_DIR / "errors_per_class.csv")

size_df = pd.DataFrame(size_rows)
size_recall = size_df.groupby(["size", "model"]).found.mean().unstack().reindex(["small", "medium", "large"])
size_count = size_df[size_df.model == size_df.model.iloc[0]].groupby("size").size().reindex(["small", "medium", "large"])
print("Number of test objects per size:", size_count.fillna(0).astype(int).to_dict())
display(size_recall.round(3))

fig, ax = plt.subplots(1, 2, figsize=(16, 4.8))
err_df[["correct (TP)", "missed (FN)", "FP: wrong class", "FP: poor localisation", "FP: duplicate", "FP: background"]].plot.bar(
    ax=ax[0], stacked=False, rot=0, color=["#55A868", "#DD8452", "#C44E52", "#8172B3", "#937860", "#DA8BC3"])
ax[0].set_title(f"Detection outcomes on the test set (conf ≥ {CONF_THR}, IoU ≥ {IOU_THR})"); ax[0].set_ylabel("count")
size_recall.plot.bar(ax=ax[1], rot=0); ax[1].set_ylim(0, 1); ax[1].set_title("Recall by object size"); ax[1].set_ylabel("recall")
plt.tight_layout(); plt.savefig(FIG_DIR / "05_error_analysis.png", dpi=150); plt.show()

### 9.2 Visual comparison on test images
Each row is one test image. The first column shows the **ground truth**, and the next columns show each model's output:
**green = correct**, **red = false detection** (the label says which type), **orange dashed = missed object**.
The images shown are the ones where the models made the most mistakes, which are the most useful for the discussion.

In [ ]:
MAX_DRAW = 25   # draw at most this many predictions per panel

def show_comparison(fnames, save_as):
    names = list(analysis)
    fig, axes = plt.subplots(len(fnames), len(names) + 1, figsize=(5 * (len(names) + 1), 4.2 * len(fnames)), squeeze=False)
    for row, fname in zip(axes, fnames):
        g = GT_TEST[fname]; img = Image.open(g["path"])
        row[0].imshow(img); row[0].set_title(f"Ground truth: {fname}", fontsize=9)
        draw_boxes(row[0], g["boxes"], [CLASS_NAMES[c] for c in g["cls"]], "#1f77b4")
        for ax, name in zip(row[1:], names):
            d = analysis[name][fname]; ax.imshow(img)
            n_err = sum(o != "TP" for o in d["outcome"]) + int((~d["matched"]).sum())
            top = np.argsort(-d["pconf"])[:MAX_DRAW]                     # keep the figure readable
            extra = f", showing top {MAX_DRAW} of {len(d['pconf'])} boxes" if len(d["pconf"]) > MAX_DRAW else ""
            ax.set_title(f"{name}  ({n_err} errors{extra})", fontsize=9)
            for b, c, cf, o in zip(d["pb"][top], d["pc"][top], d["pconf"][top], np.array(d["outcome"], dtype=object)[top]):
                lab = f"{CLASS_NAMES[c]} {cf:.2f}" + ("" if o == "TP" else f" [{o[4:]}]")
                draw_boxes(ax, [b], [lab], "#2ca02c" if o == "TP" else "#d62728")
            miss = ~d["matched"]
            draw_boxes(ax, g["boxes"][miss], [f"MISSED {CLASS_NAMES[c]}" for c in g["cls"][miss]], "#ff7f0e", ls="--")
        for ax in row:
            ax.axis("off")
    plt.tight_layout(); plt.savefig(FIG_DIR / save_as, dpi=110); plt.show()

errors_per_image = {f: sum(sum(o != "TP" for o in analysis[n][f]["outcome"]) + (~analysis[n][f]["matched"]).sum() for n in analysis)
                    for f in GT_TEST}
hardest = sorted(errors_per_image, key=errors_per_image.get, reverse=True)[:4]
show_comparison(hardest, "06_hardest_test_images.png")

Some images that went well too, since the report should show both successes and failures:

In [ ]:
easiest = [f for f in sorted(errors_per_image, key=errors_per_image.get) if len(GT_TEST[f]["cls"]) > 1][:3] or \
          sorted(errors_per_image, key=errors_per_image.get)[:3]
show_comparison(easiest, "07_good_test_images.png")

---
## 10. Final comparison

All the numbers in one table. It is saved to `results/model_comparison.csv`, and a `results/summary.md` is written that you can copy straight into the report.

In [ ]:
times = json.loads(TIMES_FILE.read_text()) if TIMES_FILE.exists() else {}
final_df = test_df.join(err_df[["correct (TP)", "missed (FN)", "false (FP)", "F1@0.25", "latency_ms"]])
final_df.insert(2, "train_min", [times.get(n, np.nan) for n in final_df.index])
final_df.to_csv(RESULTS_DIR / "model_comparison.csv")
display(final_df.round(3))

fig, ax = plt.subplots(1, 2, figsize=(16, 4.8))
final_df[["precision", "recall", "mAP50", "mAP50-95"]].plot.bar(ax=ax[0], rot=0, ylim=(0, 1))
ax[0].set_title("Test-set metrics"); ax[0].legend(loc="lower right")
for name, r in final_df.iterrows():
    ax[1].scatter(r["latency_ms"], r["mAP50-95"], s=r["params_M"] * 20, alpha=0.6, label=f"{name} ({r['params_M']:.1f}M params)")
ax[1].legend(markerscale=0.4)
ax[1].set_xlabel(f"latency per image (ms, {'GPU' if GPU else 'CPU'})"); ax[1].set_ylabel("test mAP@0.5:0.95")
ax[1].set_title("Accuracy vs speed (bubble size = model size)")
plt.tight_layout(); plt.savefig(FIG_DIR / "08_model_comparison.png", dpi=150); plt.show()

### 10.1 Automatic summary (facts for the report)
This cell writes out the key facts. The *interpretation* (why a model won, what it means for a sorting system) is for your team to write in the conclusion.

In [ ]:
def md_table(df, fmt="{:.3f}"):
    cols = [df.index.name or ""] + [str(c) for c in df.columns]
    out = ["| " + " | ".join(cols) + " |", "|" + "---|" * len(cols)]
    for idx in df.index:
        vals = [df.at[idx, c] for c in df.columns]                       # keeps each column's own type
        vals = [fmt.format(v) if isinstance(v, (float, np.floating)) else str(v) for v in vals]
        out.append("| " + " | ".join([str(idx)] + vals) + " |")
    return "\n".join(out)

best_map   = final_df["mAP50-95"].idxmax()
best_rec   = final_df["recall"].idxmax()
best_prec  = final_df["precision"].idxmax()
fastest    = final_df["latency_ms"].idxmin()
worst_cls  = ap_table.mean(axis=1).idxmin(); best_cls = ap_table.mean(axis=1).idxmax()
fp_cols    = ["FP: wrong class", "FP: poor localisation", "FP: duplicate", "FP: background"]
small      = size_recall.loc["small"].dropna() if "small" in size_recall.index else pd.Series(dtype=float)
small_line = ("- Recall on small objects: " + ", ".join(f"{m} {v:.2f}" for m, v in small.items())) if len(small) else "- No small objects in the test set."
facts = [
    f"- Dataset: {len(image_info)} original images, {len(obj_df)} objects, {len(CLASS_NAMES)} classes; split {', '.join(f'{k} {len(v)}' for k, v in splits.items())} images.",
    f"- Training: {'GPU ' + torch.cuda.get_device_name(0) if GPU else 'CPU'}, image size {IMGSZ}, batch {BATCH}, " + (f"up to {EPOCHS} epochs with early stopping (patience {PATIENCE})." if TIME_HOURS is None else f"time budget {TIME_HOURS} h per model."),
    f"- Best box quality (mAP@0.5:0.95): **{best_map}** ({final_df.loc[best_map, 'mAP50-95']:.3f}); best mAP@0.5: **{final_df['mAP50'].idxmax()}** ({final_df['mAP50'].max():.3f}).",
    f"- Highest recall (fewest misses): **{best_rec}**; highest precision (fewest false alarms): **{best_prec}**.",
    f"- Fastest inference: **{fastest}** ({final_df.loc[fastest, 'latency_ms']:.1f} ms/image).",
    f"- Easiest class on average: **{best_cls}**; hardest class: **{worst_cls}** (mean AP@0.5 {ap_table.mean(axis=1).min():.3f}).",
    small_line,
] + [f"- {m}: most common false-detection type = {err_df.loc[m, fp_cols].astype(int).idxmax()} ({int(err_df.loc[m, fp_cols].max())} of {int(err_df.loc[m, 'false (FP)'])} false detections)." for m in err_df.index]

summary = "\n\n".join([
    "# Waste & Trash Detection: results summary", "## Key facts", "\n".join(facts),
    "## Test-set comparison", md_table(final_df),
    "## AP@0.5 per class (test set)", md_table(ap_table),
    f"## Error analysis (conf ≥ {CONF_THR}, IoU ≥ {IOU_THR})", md_table(err_df, fmt="{:.3f}"),
    "## Recall by object size", md_table(size_recall),
    "## Data split (objects per class)", md_table(split_table, fmt="{:.0f}"),
])
(RESULTS_DIR / "summary.md").write_text(summary, encoding="utf-8")
display(Markdown("\n".join(facts)))
print("\nSaved ->", RESULTS_DIR / "summary.md")

---
## 11. Try the best model on your own images (live demo)
Put any photos of rubbish (e.g. taken with your phone) into the `my_test_images/` folder and run this cell. It's a handy demo for the evaluation, showing the model working on truly unseen images.

In [ ]:
MY_IMAGES = PROJECT_DIR / "my_test_images"
MY_IMAGES.mkdir(exist_ok=True)
DEMO_MODEL = best_map                        # or type a name, e.g. "YOLOv8m"

imgs = sorted(p for p in MY_IMAGES.iterdir() if p.suffix.lower() in IMG_EXT)
if not imgs:
    print(f"No images yet. Put .jpg/.png files in {MY_IMAGES} and run this cell again.")
else:
    model = load_best(DEMO_MODEL)
    cols = min(3, len(imgs)); rows_ = int(np.ceil(len(imgs[:9]) / cols))
    fig, axes = plt.subplots(rows_, cols, figsize=(6 * cols, 5 * rows_), squeeze=False)
    for ax, p in zip(axes.flat, imgs[:9]):
        r = model.predict(str(p), conf=CONF_THR, imgsz=IMGSZ, device=DEVICE, verbose=False)[0]
        ax.imshow(r.orig_img[:, :, ::-1])            # the image exactly as the model saw it (BGR -> RGB)
        draw_boxes(ax, r.boxes.xyxy.cpu().numpy(),
                   [f"{CLASS_NAMES[int(c)]} {cf:.2f}" for c, cf in zip(r.boxes.cls, r.boxes.conf)], "#2ca02c")
        ax.set_title(f"{p.name}: {len(r.boxes)} objects ({DEMO_MODEL})", fontsize=9)
    for ax in axes.flat:
        ax.axis("off")
    plt.tight_layout(); plt.show()

---
## 12. Observations & conclusion (for the team to write)

Use `results/summary.md` and the figures in `report_figures/` to answer these in your report. The numbers are produced above. **The interpretation should be your own.**

1. **Overall winner:** which model has the best mAP@0.5:0.95, and is the gap large or within noise? With only ~25 test images, one or two objects can move the score by a few points.
2. **Accuracy vs speed vs size:** is the most accurate model also the slowest? Which would you deploy on a sorting-line camera, and why?
3. **CNN vs transformer:** how did RT-DETR do compared with the two YOLOs? Link this to its data and epoch needs (see the training curves) and to having no NMS.
4. **YOLOv8m vs YOLO11m:** did the newer architecture help on a small dataset?
5. **Hard classes:** which classes had the lowest AP, and why? Consider size, number of training examples, and look-alike classes (use the confusion matrix).
6. **Error types:** were errors mostly *missed* objects or *false* detections? Which false-detection type dominated, and what does that say (wrong class = look-alike categories; background = cluttered scenes; poor localisation = unclear object edges, overlap)?
7. **Link back to the challenges in the brief:** overlap, varying shapes, sizes and backgrounds, and manual sorting.
8. **Limitations & future work:** small dataset (163 images), small test set, limited training time. Possible improvements: more images, higher resolution for small objects, class-balanced sampling, tuning the confidence threshold for each class, test-time augmentation, ensembling models.

---
## 13. Viva cheat-sheet (every team member should be able to answer these)

| Question | Short answer |
|----------|--------------|
| Why not train on the 4,923 augmented images? | They are copies of the 163 originals. Splitting them would put copies of the same photo in both train and test (**data leakage**), and the scores would come out too high. We split the originals and augment only the training images, on the fly. |
| Why a separate validation *and* test set? | Validation is used to choose the best epoch and for early stopping, so we "tune" on it. Test is touched once, at the end, which makes it an unbiased estimate on unseen data. |
| What is transfer learning? | Starting from weights pre-trained on COCO (118k images) and fine-tuning them on our data. It is essential with only 115 training images. |
| What is IoU? | Overlap area ÷ union area of two boxes. We count a detection as correct at IoU ≥ 0.5. |
| Precision vs recall? | Precision: how many detections were right (false alarms). Recall: how many real objects were found (misses). |
| mAP@0.5 vs mAP@0.5:0.95? | The first only asks "roughly the right place". The second averages over IoU 0.5 to 0.95, so it rewards tight boxes, which makes it our measure of bounding-box quality. |
| What is NMS, and why doesn't RT-DETR need it? | NMS removes overlapping duplicate boxes for the same object. RT-DETR is trained with one-to-one (Hungarian) matching, so each object gets only one prediction. |
| One-stage vs transformer detector? | YOLO predicts boxes from a dense grid in one pass and needs NMS. RT-DETR uses attention and object queries, is end-to-end, and usually needs more data and epochs. |
| What is mosaic augmentation? | It stitches 4 training images into one, so the model sees objects at new scales, positions and backgrounds, some partly hidden. It is turned off for the last 10 epochs. |
| What is an epoch, batch, image size? | Epoch = one pass over the training set. Batch = images per weight update. imgsz = the resolution the images are resized to. |
| What does early stopping (patience) do? | It stops training when validation mAP hasn't improved for `PATIENCE` epochs, which saves time and avoids overfitting. `best.pt` = the best epoch. |
| How does the confidence threshold affect results? | Higher threshold means fewer detections: precision goes up and recall goes down. mAP is independent of the threshold. The error analysis uses 0.25. |
| Why were images re-saved / rotated? | Phone photos carry an EXIF rotation tag. We make sure pixels and boxes use the same orientation, and save without the tag. |
| How would you deploy this? | Export the best model (`model.export(format="onnx")`) and run it on a camera over the conveyor. Each detected class triggers the matching sorting bin. |